# CrewAI — Analyst → Strategist crew

CrewAI models a workflow as a **crew**: you define `Agent`s with a role, a goal, and a
backstory, give them `Task`s, and pick a `Process` (*sequential* or *hierarchical*) that
decides the call order for you. You describe *who does what*, not *how the calls chain*.

| Framework | Core mental model | How multi-step flow is defined | Orchestration style |
|---|---|---|---|
| LangGraph | Explicit state graph | You wire nodes (functions/LLM calls) and edges yourself | Manual, graph-based — full control, more code |
| **CrewAI** | Role-based crew | You define `Agent`s (role/goal) and `Task`s; a `Process` (sequential/hierarchical) runs them | Opinionated, role-based — less code, less control |
| Microsoft Agent Framework | Unified successor to AutoGen + Semantic Kernel | Agents come from a chat client; you wire them with a Builder (Sequential, Concurrent, Handoff, GroupChat, Magentic) | Both — low-level like LangGraph, or pre-built patterns like CrewAI |



In [1]:
%pip install -q crewai anthropic

After the install cell finishes for the first time, use
> **Runtime ▸ Restart session** before running the rest of the notebook. Colab can keep
> an old, incompatible copy of a dependency loaded in memory even after a fresh install
> replaces it on disk.


In [9]:
import os

def _load(key, default=None):
  """Colab Secrets first, then environment variable, then interactive prompt."""
  try:
    from google.colab import userdata
    val = userdata.get(key)
    if val:
      return val
  except Exception:
    pass
  val = os.environ.get(key, default)
  if not val:
    from getpass import getpass
    val = getpass(f"{key}: ")
  return val

ANTHROPIC_API_KEY = _load("ANTHROPIC_API_KEY")
ANTHROPIC_MODEL = _load("ANTHROPIC_MODEL", "claude-sonnet-4-5-20250929")
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

print(f"Using model: {ANTHROPIC_MODEL}")


Using model: claude-sonnet-4-5-20250929


## The scenario

An assistant reads user interviews, support tickets, and
competitor notes, and is supposed to turn that into a feature recommendation — without
mismatched analysis or hallucinated suggestions.

We'll build that as a **two-step pipeline**:

1. **Analyst** — reads the raw signals, extracts 2–3 themes, and cites which signal(s)
   support each one (a small guardrail against fabrication).
2. **Strategist** — takes *only* the Analyst's themes and recommends ONE feature with a
   one-line rationale. It's told not to invent anything the Analyst didn't surface.


In CrewAI terms: two **Agents** (Analyst, Strategist) doing the work. We run them two ways to show the difference:

| | **Version A: Sequential** | **Version B: Hierarchical** |
|---|---|---|
| Who decides the order? | You: tasks run top to bottom | A **manager Agent** plans and delegates |
| Who does each task? | You assign `agent=` on every Task | The manager picks a coworker for each step |
| Tasks you write | Two step-by-step Tasks | One goal-level Task |
| Quality control | None built in | The manager reviews each answer and can send work back |
| Cost | 2 LLM calls (one per agent) | More calls: manager planning, delegation and review |


In [10]:
SIGNALS = """
- User interview (Maria, SMB owner): "I got lost during setup and gave up before finishing."
- Support ticket #4821: "Can't find where to invite my team members. Confusing menu."
- Competitor teardown (RivalApp): just shipped a guided setup checklist with a progress bar.
- Support ticket #4903: "Wish there was a quick tour when I first logged in."
""".strip()

print(SIGNALS)


- User interview (Maria, SMB owner): "I got lost during setup and gave up before finishing."
- Support ticket #4821: "Can't find where to invite my team members. Confusing menu."
- Competitor teardown (RivalApp): just shipped a guided setup checklist with a progress bar.
- Support ticket #4903: "Wish there was a quick tour when I first logged in."


In [11]:
from crewai import Agent, LLM

# max_iter gives the agent room to retry within a turn.
llm = LLM(model=f"anthropic/{ANTHROPIC_MODEL}", max_tokens=400)

analyst = Agent(
  role="Product Signal Analyst",
  goal="Extract 2-3 themes from raw customer signals, citing sources",
  backstory="You read user interviews, tickets, and competitor notes for a living.",
  llm=llm,
  max_iter=15,
  allow_delegation=False,  # workers do their own job; only the manager delegates (Version B)
  verbose=False,
)

strategist = Agent(
  role="Product Strategist",
  goal="Recommend ONE feature with a one-line rationale, grounded only in the themes",
  backstory="You turn the analyst's themes into a single prioritized call.",
  llm=llm,
  max_iter=15,
  allow_delegation=False,
  verbose=False,
)

## Version A: Sequential process

You decide everything: each Task names its `agent`, and the Tasks run in list order. The
second Task gets the first one's output through `context`.

In [16]:
from crewai import Task, Crew, Process

# Retrieve workspace ID from environment or Colab Secrets
ANTHROPIC_WORKSPACE_ID = _load("ANTHROPIC_WORKSPACE_ID", "")
print("\n\n")

# Re-initialize LLM with workspace header if available to bypass the 400 error
if ANTHROPIC_WORKSPACE_ID:
    llm.extra_headers = {"anthropic-workspace-id": ANTHROPIC_WORKSPACE_ID}

analyze_task = Task(
  description="Signals:\n{signals}\n\nExtract 2-3 themes as bullet points, citing which signal(s) support each.",
  expected_output="A short bullet list of themes with citations.",
  agent=analyst,
)

recommend_task = Task(
  description="Using ONLY the themes above, recommend ONE feature to prioritize next sprint with a one-line rationale.",
  expected_output="One feature recommendation with a one-line rationale.",
  agent=strategist,
  context=[analyze_task],
)

crew_seq = Crew(
  agents=[analyst, strategist],
  tasks=[analyze_task, recommend_task],
  process=Process.sequential,
  verbose=False,
)

result_seq = await crew_seq.kickoff_async(inputs={"signals": SIGNALS})
print(result_seq)

ANTHROPIC_WORKSPACE_ID: ··········



# Feature Recommendation

**Feature: Guided Onboarding Checklist with Progress Tracking**

**One-line Rationale:** A guided setup experience directly addresses user drop-off during onboarding while simultaneously improving feature discoverability, closing the competitive gap identified with RivalApp's recent launch.


## Version B: Hierarchical process with a manager Agent

Three changes from Version A:

1. **A manager Agent** (`allow_delegation=True`, no tools) is passed as `manager_agent=`.
   It is *not* listed in `agents=`; it supervises them.
2. **One goal-level Task with no `agent=`.** In hierarchical mode, every Task is handed to
   the manager. If you pinned `agent=analyst` on a Task, the manager could only delegate to
   that one agent. Leaving it off lets the manager choose. With a single goal (rather than
   two pre-ordered steps), the manager also has to work out that the analysis comes first.
3. `process=Process.hierarchical`.

CrewAI gives the manager two built-in tools: **Delegate work to coworker** and
**Ask question to coworker**. The `step_callback` below logs every time the manager uses
one, so you can see the call order it chose.

> **How "deciding the order" actually works:** CrewAI still processes the `tasks` list from
> top to bottom. The manager decides *who* handles each Task, *how many* delegations it
> takes, and whether to send work back. That's why this version uses one goal-level Task:
> it gives the manager room to plan the steps itself.
>
> **Simpler alternative:** use `manager_llm=llm` instead of `manager_agent=`, and CrewAI
> builds a generic manager for you. A custom manager lets you write its rules, such as
> "make sure the recommendation is grounded."

In [17]:
# The manager gets more room than the workers: it writes plans, delegation
# instructions and reviews, so 400 tokens can cut it off mid-thought --> 1200 is more apt.
manager_llm = LLM(model=f"anthropic/{ANTHROPIC_MODEL}", max_tokens=1200)

manager = Agent(
  role="Product Team Lead",
  goal="Deliver ONE well-grounded feature recommendation by coordinating your team",
  backstory=(
    "You lead a product squad with a Product Signal Analyst and a Product Strategist. "
    "You never do the analysis or write the recommendation yourself: you decide who "
    "should work on what, in what order, and you check every answer before accepting it. "
    "You reject any recommendation that relies on evidence not present in the original signals."
  ),
  llm=manager_llm,
  allow_delegation=True,  # required: delegation is the manager's whole job
  max_iter=15,
  verbose=False,
)

In [18]:
import re

try:  # newer CrewAI
  from crewai.events import crewai_event_bus, ToolUsageStartedEvent
except ImportError:  # older CrewAI
  from crewai.utilities.events import crewai_event_bus, ToolUsageStartedEvent

delegation_log = []

@crewai_event_bus.on(ToolUsageStartedEvent)
def _log_delegation(source, event):
  if "coworker" not in str(event.tool_name).lower():
    return
  args = event.tool_args
  who = args.get("coworker") if isinstance(args, dict) else None
  if not who:
    m = re.search(r"""['"]coworker['"]\s*:\s*['"]([^'"]+)""", str(args))
    who = m.group(1) if m else "?"
  delegation_log.append((event.tool_name, who))
  print(f"[manager] step {len(delegation_log)}: {event.tool_name}  ->  {who}")


goal_task = Task(
  description=(
    "Signals:\n{signals}\n\n"
    "Turn these customer signals into ONE feature to prioritize next sprint. "
    "The recommendation must be grounded in themes extracted from the signals, with "
    "each theme citing the signal(s) that support it. Do not invent new signals."
  ),
  expected_output=(
    "The 2-3 themes with citations, then ONE feature recommendation with a one-line rationale."
  ),
)

crew_hier = Crew(
  agents=[analyst, strategist],  # the workers (manager is NOT listed here)
  tasks=[goal_task],
  process=Process.hierarchical,
  manager_agent=manager,
  verbose=False,
)

result_hier = await crew_hier.kickoff_async(inputs={"signals": SIGNALS})
print("\nFINAL OUTPUT:\n", result_hier)

[manager] step 1: delegate_work_to_coworker  ->  Product Signal Analyst
[manager] step 2: delegate_work_to_coworker  ->  Product Strategist

FINAL OUTPUT:
 Perfect! Let me compile the complete final answer with the themes and feature recommendation:

---

## **THEMES WITH CITATIONS**

**Theme 1: Onboarding Experience is Incomplete/Inadequate**
- Signal 1 (User interview - Maria, SMB owner): "I got lost during setup and gave up before finishing."
- Signal 4 (Support ticket #4903): "Wish there was a quick tour when I first logged in."

**Theme 2: Poor Information Architecture/Navigation**
- Signal 2 (Support ticket #4821): "Can't find where to invite my team members. Confusing menu."

**Theme 3: Competitive Gap in Setup Guidance Features**
- Signal 3 (Competitor teardown - RivalApp): just shipped a guided setup checklist with a progress bar.
- Signal 1 (User interview - Maria, SMB owner): "I got lost during setup and gave up before finishing."
- Signal 4 (Support ticket #4903): "Wish the

## Comparing the two runs

Both versions should land on a similar recommendation. The difference is in *how* they got
there and what it cost. In the delegation log, check whether the manager went to the Analyst
first: in Version A you guaranteed that order, while in Version B the manager had to work it out.

In [19]:
def usage(crew):
  m = getattr(crew, "usage_metrics", None)
  if m is None:
    return "n/a", "n/a"
  return getattr(m, "successful_requests", "n/a"), getattr(m, "total_tokens", "n/a")

seq_calls, seq_tokens = usage(crew_seq)
hier_calls, hier_tokens = usage(crew_hier)

print(f"{'':<28}{'Sequential':>14}{'Hierarchical':>16}")
print(f"{'Who set the call order':<28}{'you':>14}{'manager':>16}")
print(f"{'Manager delegations':<28}{'-':>14}{len(delegation_log):>16}")
print(f"{'LLM requests':<28}{str(seq_calls):>14}{str(hier_calls):>16}")
print(f"{'Total tokens':<28}{str(seq_tokens):>14}{str(hier_tokens):>16}")

print("\nManager's chosen call order:")
for i, (tool, who) in enumerate(delegation_log, 1):
  print(f"  {i}. {tool} -> {who}")

                                Sequential    Hierarchical
Who set the call order                 you         manager
Manager delegations                      -               2
LLM requests                            12              19
Total tokens                          6208           15260

Manager's chosen call order:
  1. delegate_work_to_coworker -> Product Signal Analyst
  2. delegate_work_to_coworker -> Product Strategist


Microsoft Agent Framework's `GroupChatBuilder`/`MagenticBuilder` patterns aim at similar
"describe the team, not the wiring" territory, but layered on enterprise features
(sessions, middleware, OpenTelemetry) CrewAI doesn't focus on.

**When to use which:** use **sequential** when you already know the steps, as in this
two-step hand-off. It's cheaper, faster and predictable. Use **hierarchical** when the path
depends on the content: open-ended goals, several possible specialists, or when you want a
supervisor reviewing work before it ships. You pay for that flexibility in extra LLM calls
and less predictable runs.


